# Mini-Project 02: Will This Be a Strong Sales Week?

## Part P — Probability of each event

In [15]:
import numpy as np, pandas as pd
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import confusion_matrix, precision_score, recall_score, accuracy_score

df = pd.read_csv("Walmart.csv")
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
print(len(df), df["Store"].nunique(), df["Date"].nunique(), df["Holiday_Flag"].sum())

def add_events(d, store_avg):
    d = d.copy()
    avg = d["Store"].map(store_avg)                      
    d["S"] = d["Weekly_Sales"] >= 1.10 * avg             
    d["H"] = d["Holiday_Flag"] == 1
    d["Q"] = d["Date"].dt.month.isin([11, 12])
    d["C"] = d["Temperature"] < 40
    d["F"] = d["Fuel_Price"] >= 3.50
    d["J"] = d["Unemployment"] >= 8
    d["G"] = avg >= 1_000_000                           
    return d

6435 45 143 450


In [16]:
ev = add_events(df, df.groupby("Store")["Weekly_Sales"].mean())
rows = []
for e in "SHQCFJG":
    rows.append((e, int(ev[e].sum()), round(ev[e].mean(), 4), round(1 - (~ev[e]).mean(), 4),
                 round(ev[e].mean() + (~ev[e]).mean(), 4)))
print(pd.DataFrame(rows, columns=["event", "count", "P(E)", "1-P(Ec)", "P(E)+P(Ec)"]).to_string(index=False))

for a, b in [("S","H"), ("S","Q"), ("H","Q"), ("S","C")]:
    n = int((ev[a] & ev[b]).sum())
    print(f"P({a} and {b}) = {n}/{len(ev)} = {n/len(ev):.4f}")

event  count   P(E)  1-P(Ec)  P(E)+P(Ec)
    S    814 0.1265   0.1265         1.0
    H    450 0.0699   0.0699         1.0
    Q    810 0.1259   0.1259         1.0
    C    989 0.1537   0.1537         1.0
    F   3029 0.4707   0.4707         1.0
    J   3022 0.4696   0.4696         1.0
    G   3146 0.4889   0.4889         1.0
P(S and H) = 122/6435 = 0.0190
P(S and Q) = 333/6435 = 0.0517
P(H and Q) = 180/6435 = 0.0280
P(S and C) = 185/6435 = 0.0287


## Part B — Conditional probability and Bayes' theorem

In [ ]:
# B1
pS, pH = ev.S.mean(), ev.H.mean()
pS_H, pS_nH = ev.loc[ev.H, "S"].mean(), ev.loc[~ev.H, "S"].mean()
print(f"B1  P(S|H) = {pS_H:.4f}   P(S|Hc) = {pS_nH:.4f}   ratio = {pS_H/pS_nH:.2f}x")

# B2
pH_S_bayes = pS_H * pH / pS
pH_S_count = ev.loc[ev.S, "H"].mean()
print(f"B2  P(H|S) by Bayes = {pH_S_bayes:.4f}   by counting = {pH_S_count:.4f}")

# B3 
def group(r):
    if r.Holiday_Flag == 0: return "No holiday"
    return {2: "Super Bowl", 9: "Labor Day", 11: "Thanksgiving", 12: "Christmas"}[r.Date.month]
ev["g"] = ev.apply(group, axis=1)
t = ev.groupby("g").agg(n=("S", "size"), strong=("S", "sum"), P_S_given_g=("S", "mean"))
t["P(g)"] = t.n / len(ev)
t["P(g|S)"] = t.strong / ev.S.sum()
t["P(S|g)P(g)"] = t.P_S_given_g * t["P(g)"]
print(t.round(4).loc[["Super Bowl", "Labor Day", "Thanksgiving", "Christmas", "No holiday"]])
print("rebuilt P(S) =", round(t["P(S|g)P(g)"].sum(), 4), "| direct P(S) =", round(pS, 4))
print("highest P(S|g):", t.P_S_given_g.idxmax(), "| largest P(g|S):", t["P(g|S)"].idxmax(),
      "| largest holiday share of strong weeks:", t.drop("No holiday")["P(g|S)"].idxmax())

# B4 
pS_Q, pQ_S = ev.loc[ev.Q, "S"].mean(), ev.loc[ev.S, "Q"].mean()
print(f"B4  P(S|Q) = {pS_Q:.4f}   P(Q|S) = {pQ_S:.4f}")

B1  P(S|H) = 0.2711   P(S|Hc) = 0.1156   ratio = 2.34x
B2  P(H|S) by Bayes = 0.1499   by counting = 0.1499
                 n  strong  P_S_given_g    P(g)  P(g|S)  P(S|g)P(g)
g                                                                  
Super Bowl     135      26       0.1926  0.0210  0.0319      0.0040
Labor Day      135      16       0.1185  0.0210  0.0197      0.0025
Thanksgiving    90      74       0.8222  0.0140  0.0909      0.0115
Christmas       90       6       0.0667  0.0140  0.0074      0.0009
No holiday    5985     692       0.1156  0.9301  0.8501      0.1075
rebuilt P(S) = 0.1265 | direct P(S) = 0.1265
highest P(S|g): Thanksgiving | largest P(g|S): No holiday | largest holiday share of strong weeks: Thanksgiving
B4  P(S|Q) = 0.4111   P(Q|S) = 0.4091


In [17]:
# B5
pS = ev.S.mean()
def rule_stats(flag, name):
    p_f_S, p_f_nS = ev.loc[ev.S, flag].mean(), ev.loc[~ev.S, flag].mean()
    prec_bayes = p_f_S * pS / (p_f_S * pS + p_f_nS * (1 - pS))
    prec_count = ev.loc[ev[flag], "S"].mean()
    print(f"{name}: precision (Bayes) {prec_bayes:.4f} | by counting {prec_count:.4f} | recall {p_f_S:.4f} | misses {1-p_f_S:.1%} of strong weeks")
    return p_f_S, p_f_nS
pHS, pHnS = rule_stats("H", "Holiday rule  ")
rule_stats("Q", "Nov-Dec rule  ")

# B6
for k in (2, 0.5):
    ps = pS * k
    print(f"B6  P(S) x{k}: P(S) = {ps:.4f} -> holiday-rule precision = {pHS*ps/(pHS*ps + pHnS*(1-ps)):.4f}")

# B7 
rows = []
for e in "HQCFJG":
    j = (ev.S & ev[e]).mean(); prod = ev.S.mean() * ev[e].mean()
    rows.append((e, round(j,4), round(prod,4), round(j/prod,2), round(ev.loc[ev[e],"S"].mean(),4), round(ev.loc[~ev[e],"S"].mean(),4)))
print(pd.DataFrame(rows, columns=["E", "P(S and E)", "P(S)P(E)", "ratio", "P(S|E)", "P(S|Ec)"]).to_string(index=False))

Holiday rule  : precision (Bayes) 0.2711 | by counting 0.2711 | recall 0.1499 | misses 85.0% of strong weeks
Nov-Dec rule  : precision (Bayes) 0.4111 | by counting 0.4111 | recall 0.4091 | misses 59.1% of strong weeks
B6  P(S) x2: P(S) = 0.2530 -> holiday-rule precision = 0.4652
B6  P(S) x0.5: P(S) = 0.0632 -> holiday-rule precision = 0.1478
E  P(S and E)  P(S)P(E)  ratio  P(S|E)  P(S|Ec)
H      0.0190    0.0088   2.14  0.2711   0.1156
Q      0.0517    0.0159   3.25  0.4111   0.0855
C      0.0287    0.0194   1.48  0.1871   0.1155
F      0.0452    0.0595   0.76  0.0961   0.1536
J      0.0640    0.0594   1.08  0.1363   0.1178
G      0.0570    0.0618   0.92  0.1167   0.1359


**B7 reading.** Q (ratio 3.25) and H (2.14) carry the most information about strong weeks; C (1.48) a little; F is *negatively* related (0.76); J (1.08) and G (0.92) are close to independent. Caution: F and J move with the calendar (fuel prices and unemployment trend over 2010–2012), so they may stand in for time or season rather than cause anything.

## Part N — Naive Bayes (train 2010–2011, test 2012)

In [ ]:
# N1 
train_raw, test_raw = df[df.Date.dt.year <= 2011], df[df.Date.dt.year == 2012]
store_avg = train_raw.groupby("Store")["Weekly_Sales"].mean()
train, test = add_events(train_raw, store_avg), add_events(test_raw, store_avg)
print(f"N1  train n={len(train)}  P(S)={train.S.mean():.4f}   test n={len(test)}  P(S)={test.S.mean():.4f}")

# N2 
X = ["H", "Q", "C", "F", "J", "G"]
prior = train.S.mean()
print(f"priors: P(S)={prior:.4f}  P(Sc)={1-prior:.4f}")
lik = pd.DataFrame({"P(f=1|S)": [train.loc[train.S, f].mean() for f in X],
                    "P(f=1|Sc)": [train.loc[~train.S, f].mean() for f in X]}, index=X)
print(lik.round(4))

N1  train n=4500  P(S)=0.1276   test n=1935  P(S)=0.1318
priors: P(S)=0.1276  P(Sc)=0.8724
   P(f=1|S)  P(f=1|Sc)
H    0.1725     0.0665
Q    0.5836     0.1210
C    0.2875     0.1526
F    0.1882     0.3459
J    0.5523     0.5311
G    0.4634     0.4926


In [ ]:
# N3 
profA = dict(H=1, Q=1, C=1, F=0, J=0, G=1)  
profB = dict(H=0, Q=0, C=0, F=1, J=1, G=0)   
def posterior(x, alpha=0):
    n1, n0 = train.S.sum(), (~train.S).sum()
    s1, s0 = prior, 1 - prior
    for f in X:
        p1 = (train.loc[train.S, f].sum() + alpha) / (n1 + 2*alpha)
        p0 = (train.loc[~train.S, f].sum() + alpha) / (n0 + 2*alpha)
        s1 *= p1 if x[f] else 1 - p1
        s0 *= p0 if x[f] else 1 - p0
    return s1, s0, s1 / (s1 + s0)
for nm, x in [("a", profA), ("b", profB)]:
    s1, s0, p = posterior(x)
    print(f"N3 profile ({nm}): score(S)={s1:.6f}  score(Sc)={s0:.6f}  P(S|x)={p:.4f}   [alpha=1: {posterior(x,1)[2]:.4f}]")

# N4 
trS = train[train.S]
print(f"N4  given S: P(H and Q|S)={(trS.H & trS.Q).mean():.4f}  vs  P(H|S)P(Q|S)={trS.H.mean()*trS.Q.mean():.4f}")

# N5
nb = CategoricalNB(alpha=1, min_categories=2).fit(train[X].astype(int), train["S"])
for nm, x in [("a", profA), ("b", profB)]:
    print(f"N5  profile ({nm}): sklearn P(S|x) = {nb.predict_proba(pd.DataFrame([x])[X])[0,1]:.4f}")

N3 profile (a): score(S)=0.000622  score(Sc)=0.000162  P(S|x)=0.7935   [alpha=1: 0.7939]
N3 profile (b): score(S)=0.001746  score(Sc)=0.056546  P(S|x)=0.0300   [alpha=1: 0.0301]
N4  given S: P(H and Q|S)=0.1394  vs  P(H|S)P(Q|S)=0.1007
N5  profile (a): sklearn P(S|x) = 0.7939
N5  profile (b): sklearn P(S|x) = 0.0301


In [ ]:
# N6 
def report(pred, name):
    cm = confusion_matrix(test.S, pred, labels=[False, True])
    return dict(model=name, TN=cm[0,0], FP=cm[0,1], FN=cm[1,0], TP=cm[1,1],
                accuracy=round(accuracy_score(test.S, pred), 4),
                precision=round(precision_score(test.S, pred, zero_division=0), 4),
                recall=round(recall_score(test.S, pred), 4))
pred_nb = nb.predict(test[X].astype(int))
print(pd.DataFrame([report(pred_nb, "Naive Bayes (t=0.5)"),
                    report(np.zeros(len(test), bool), "never strong"),
                    report(test.H.values, "strong if holiday")]).to_string(index=False))

proba = nb.predict_proba(test[X].astype(int))[:, 1]
print(f"\nmax predicted P(S|x) on 2012 = {proba.max():.3f}")
print("2012 weeks in Nov-Dec:", int(test.Q.sum()), "| 2012 holiday weeks:", int(test.H.sum()), "(Super Bowl, Labor Day only)")
print("P(S|H) train =", round(train.loc[train.H,"S"].mean(),3), "| test =", round(test.loc[test.H,"S"].mean(),3))

# N7 
print(pd.DataFrame([report(proba > t, f"t={t}") for t in (0.1, 0.2, 0.3, 0.5)]).to_string(index=False))

              model   TN  FP  FN  TP  accuracy  precision  recall
Naive Bayes (t=0.5) 1680   0 255   0    0.8682     0.0000  0.0000
       never strong 1680   0 255   0    0.8682     0.0000  0.0000
  strong if holiday 1613  67 232  23    0.8455     0.2556  0.0902

max predicted P(S|x) on 2012 = 0.317
2012 weeks in Nov-Dec: 0 | 2012 holiday weeks: 90 (Super Bowl, Labor Day only)
P(S|H) train = 0.275 | test = 0.256
model   TN  FP  FN  TP  accuracy  precision  recall
t=0.1 1574 106 236  19    0.8233      0.152  0.0745
t=0.2 1673   7 252   3    0.8661      0.300  0.0118
t=0.3 1678   2 255   0    0.8672      0.000  0.0000
t=0.5 1680   0 255   0    0.8682      0.000  0.0000


## Findings

- **P:** From the dataset, I found that 12.65% of the weeks had strong sales (814 out of 6,435). About 7% were holiday weeks, and 12.6% were in November or December.

- **B:** The chance of having a strong week during a holiday was 27.1%, compared to 11.6% during non-holiday weeks. I also got 15% for P(H|S) using both Bayes' theorem and counting. Thanksgiving had the highest chance of strong sales at 82.2%, but 85% of strong weeks still happened outside holidays. Christmas was actually below average at 6.7%.

- **Rules:** The holiday staffing rule had 27.1% precision and 15% recall, meaning it missed 85% of strong weeks. The November–December rule worked better, with 41.1% precision and 40.9% recall. I also found that if strong weeks were twice as common, the holiday rule's precision would increase to 46.5%, but if they were half as common, it would drop to 14.8%.

- **N:** For Naive Bayes, I got a 79% chance of strong sales for profile (a) and 3% for profile (b), which matched the scikit-learn results. However, the model did not perform well on the 2012 test data. Its highest predicted probability was only 32%, so using a 0.5 threshold, it did not predict any strong weeks. It still had 86.8% accuracy, but its recall was 0%. The holiday rule performed slightly better at identifying strong weeks, with 25.6% precision and 9% recall. Lowering the model's threshold to 0.1 gave 15.2% precision and 7.5% recall, which was still worse than the holiday rule. One possible reason is that the 2012 test data did not include November or December, which were important months for predicting strong sales.